# Kaggle S6E3 — Customer Churn v5
4-model ensemble: LGB (×3 seeds) + XGB (×3 seeds) + CatBoost (GPU) + NN (embeddings).
Selective n-gram interactions, TE stats (mean/std/min/max), COBYLA ensemble optimizer, 10-fold CV.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import xgboost as xgb
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')
SEED = 42; N_SPLITS = 10
np.random.seed(SEED)
print('ok')

In [ ]:
import os
if os.path.exists('/kaggle/input'):
    TRAIN_PATH = '/kaggle/input/competitions/playground-series-s6e3/train.csv'
    TEST_PATH  = '/kaggle/input/competitions/playground-series-s6e3/test.csv'
    SUB_PATH   = '/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv'
    ORIG_PATH  = '/kaggle/input/datasets/cdeotte/s6e3-original-dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv'
else:
    TRAIN_PATH = 'data/train.csv'
    TEST_PATH  = 'data/test.csv'
    SUB_PATH   = 'data/sample_submission.csv'
    ORIG_PATH  = 'data/WA_Fn-UseC_-Telco-Customer-Churn.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
orig  = pd.read_csv(ORIG_PATH)
sub   = pd.read_csv(SUB_PATH)
print('train', train.shape, '| test', test.shape, '| orig', orig.shape)

In [ ]:
TARGET='Churn'
CATS=['gender','SeniorCitizen','Partner','Dependents','PhoneService',
    'MultipleLines','InternetService','OnlineSecurity','OnlineBackup','DeviceProtection',
    'TechSupport','StreamingTV','StreamingMovies','Contract','PaperlessBilling','PaymentMethod']
SERVICE_COLS = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

train_ids=train['id'].copy(); test_ids=test['id'].copy()
train=train.drop(columns=['id']); test=test.drop(columns=['id'])
if 'customerID' in orig.columns: orig=orig.drop(columns=['customerID'])

def map_target(s): return s.map({'Yes':1,'No':0}).astype('int8') if s.dtype==object else s.astype('int8')
train[TARGET]=map_target(train[TARGET]); orig[TARGET]=map_target(orig[TARGET])

for df in [train,test,orig]:
    df['tenure']=pd.to_numeric(df['tenure'],errors='coerce').astype('float32')
    df['MonthlyCharges']=pd.to_numeric(df['MonthlyCharges'],errors='coerce').astype('float32')
    df['TotalCharges']=pd.to_numeric(df['TotalCharges'].astype(str).str.strip().replace('',np.nan),errors='coerce').astype('float32')
    m=df['TotalCharges'].isna()
    df.loc[m,'TotalCharges']=df.loc[m,'tenure']*df.loc[m,'MonthlyCharges']
    for c in CATS:
        if c in df.columns: df[c]=df[c].astype(str).fillna('Missing')

BASE=[c for c in train.columns if c!=TARGET]
print('BASE cols:',len(BASE),'| churn rate:',round(train[TARGET].mean(),4))

In [ ]:
from itertools import combinations

# ── DIGIT features ────────────────────────────────────────────
def add_digit_cols(df, col, multiplier):
    if multiplier == 'direct':
        vals = pd.to_numeric(df[col], errors='coerce').fillna(-1).astype(int)
    else:
        vals = (pd.to_numeric(df[col], errors='coerce') * multiplier).round(0).fillna(-1).astype(int)
    max_len = {'tenure':3,'MonthlyCharges':5,'TotalCharges':6}.get(col, vals.astype(str).str.len().max())
    s = vals.astype(str).str.zfill(max_len)
    created = []
    for i in range(max_len):
        n = f'{col}_DIGIT_{i+1}'; df[n] = s.str[i].astype(int); created.append(n)
    return created

DIGIT = []
for df in [train, test]:
    d = add_digit_cols(df, 'tenure', 'direct')
    add_digit_cols(df, 'MonthlyCharges', 100)
    add_digit_cols(df, 'TotalCharges', 1)
    if df is train:
        DIGIT = d + [f'MonthlyCharges_DIGIT_{i+1}' for i in range(5)] + [f'TotalCharges_DIGIT_{i+1}' for i in range(6)]
print(len(DIGIT), 'DIGIT features')

# ── ROUND features ────────────────────────────────────────────
ROUND = []
for col, levels in [('MonthlyCharges',[('1s',0),('10s',-1),('100s',-2),('1000s',-3)]),
                    ('TotalCharges',  [('1s',0),('10s',-1),('100s',-2),('1000s',-3)]),
                    ('tenure',        [('1s',0),('10s',-1)])]:
    for suffix, level in levels:
        n = f'{col}_ROUND_{suffix}'; ROUND.append(n)
        for df in [train, test]: df[n] = df[col].round(level).fillna(-1).astype(int)
print(len(ROUND), 'ROUND features')

# ── ORIG dataset stats ────────────────────────────────────────
ORIG_FEATS = []
orig_mean = float(orig[TARGET].mean())
for col in BASE:
    mm = orig.groupby(col)[TARGET].mean()
    cm = orig.groupby(col).size()
    for df in [train, test]:
        df[f'orig_mean_{col}']  = df[col].map(mm).fillna(orig_mean)
        df[f'orig_count_{col}'] = df[col].map(cm).fillna(0)
    ORIG_FEATS += [f'orig_mean_{col}', f'orig_count_{col}']
print(len(ORIG_FEATS), 'ORIG features')

# ── ARITHMETIC interactions (3 features) ─────────────────────
ARITH = ['charges_deviation', 'monthly_to_total_ratio', 'avg_monthly_charges']
for df in [train, test]:
    df['charges_deviation'] = df['TotalCharges'] - df['tenure'] * df['MonthlyCharges']
    df['monthly_to_total_ratio'] = df['MonthlyCharges'] / (df['TotalCharges'] + 1)
    df['avg_monthly_charges'] = df['TotalCharges'] / (df['tenure'] + 1)
print(len(ARITH), 'ARITH features')

# ── SERVICE counts (3 features) ──────────────────────────────
SERVICE = ['service_yes_count', 'has_internet', 'has_phone']
for df in [train, test]:
    df['service_yes_count'] = sum((df[c] == 'Yes').astype(int) for c in SERVICE_COLS)
    df['has_internet'] = (df['InternetService'] != 'No').astype(int)
    df['has_phone'] = (df['PhoneService'] == 'Yes').astype(int)
print(len(SERVICE), 'SERVICE features')

# ── FREQUENCY encoding (3 features) ──────────────────────────
FREQ = []
all_data = pd.concat([train, test, orig], ignore_index=True)
for col in ['tenure', 'MonthlyCharges', 'TotalCharges']:
    fname = f'{col}_freq'
    FREQ.append(fname)
    freq_map = all_data[col].value_counts(normalize=True)
    for df in [train, test]:
        df[fname] = df[col].map(freq_map).fillna(0).astype('float32')
print(len(FREQ), 'FREQ features')

# ── NUMCAT: numericals as categories for TE ──────────────────
NUMCAT = ['tenure_cat', 'MC_cat', 'TC_cat']
for df in [train, test]:
    df['tenure_cat'] = df['tenure'].astype(str)
    df['MC_cat'] = df['MonthlyCharges'].astype(str)
    df['TC_cat'] = df['TotalCharges'].astype(str)
print(len(NUMCAT), 'NUMCAT features')

# ── SELECTIVE N-GRAM INTERACTIONS (19 features) ──────────────
TOP6 = ['Contract', 'InternetService', 'PaymentMethod',
        'OnlineSecurity', 'TechSupport', 'PaperlessBilling']
TOP4 = TOP6[:4]
INTER = []
for c1, c2 in combinations(TOP6, 2):  # 15 bigrams
    n = f'{c1}_{c2}'; INTER.append(n)
    for df in [train, test]: df[n] = df[c1].astype(str) + '_' + df[c2].astype(str)
for c1, c2, c3 in combinations(TOP4, 3):  # 4 trigrams
    n = f'{c1}_{c2}_{c3}'; INTER.append(n)
    for df in [train, test]: df[n] = df[c1].astype(str) + '_' + df[c2].astype(str) + '_' + df[c3].astype(str)
print(len(INTER), 'INTER features (selective n-grams)')

# ── Combine TE columns and final feature list ────────────────
TE_COLS = INTER + NUMCAT  # 19 + 3 = 22 columns to target-encode
FEATURES = list(dict.fromkeys(BASE + ORIG_FEATS + ARITH + SERVICE + FREQ + TE_COLS + ROUND + DIGIT))
print(len(TE_COLS), 'TE_COLS |', len(FEATURES), 'total features')

In [ ]:
import time
from sklearn.model_selection import KFold

def factorize3(tr, val, te):
    codes, _ = pd.factorize(pd.concat([tr, val, te], ignore_index=True).astype(str))
    n1, n2 = len(tr), len(val)
    return (pd.Series(codes[:n1], index=tr.index, dtype='int32'),
            pd.Series(codes[n1:n1+n2], index=val.index, dtype='int32'),
            pd.Series(codes[n1+n2:], index=te.index, dtype='int32'))

# Precompute OOF TE with stats (mean/std/min/max) for 22 TE columns
print(f'Precomputing OOF target encoding (mean/std/min/max) for {len(TE_COLS)} columns...')
t0 = time.time()
X_all     = train[FEATURES].copy()
y_all     = train[TARGET].copy()
Xtest_raw = test[FEATURES].copy()
gm        = float(y_all.mean())
g_std     = float(y_all.std())

te_suffixes = ['mean', 'std', 'min', 'max']
oof_te = pd.DataFrame(index=X_all.index)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for fold_i, (ti, vi) in enumerate(kf.split(X_all), 1):
    Xt = X_all.iloc[ti]; yt = y_all.iloc[ti]; Xv = X_all.iloc[vi]
    fm = float(yt.mean()); f_std = float(yt.std())
    for col in TE_COLS:
        grp = Xt.assign(_y=yt.values).groupby(col)['_y']
        mp_mean = grp.mean()
        mp_std  = grp.std()
        mp_min  = grp.min()
        mp_max  = grp.max()
        oof_te.loc[Xv.index, f'TE_mean_{col}'] = Xv[col].map(mp_mean).fillna(fm).values
        oof_te.loc[Xv.index, f'TE_std_{col}']  = Xv[col].map(mp_std).fillna(f_std).values
        oof_te.loc[Xv.index, f'TE_min_{col}']  = Xv[col].map(mp_min).fillna(fm).values
        oof_te.loc[Xv.index, f'TE_max_{col}']  = Xv[col].map(mp_max).fillna(fm).values
    print(f'  TE fold {fold_i}/5 done — {time.time()-t0:.0f}s elapsed')

print('Computing test TE from full training data...')
test_te = pd.DataFrame(index=Xtest_raw.index)
for col in TE_COLS:
    grp = X_all.assign(_y=y_all.values).groupby(col)['_y']
    mp_mean = grp.mean()
    mp_std  = grp.std()
    mp_min  = grp.min()
    mp_max  = grp.max()
    test_te[f'TE_mean_{col}'] = Xtest_raw[col].map(mp_mean).fillna(gm)
    test_te[f'TE_std_{col}']  = Xtest_raw[col].map(mp_std).fillna(g_std)
    test_te[f'TE_min_{col}']  = Xtest_raw[col].map(mp_min).fillna(gm)
    test_te[f'TE_max_{col}']  = Xtest_raw[col].map(mp_max).fillna(gm)

# Drop raw TE_COLS strings, add TE stat features
X_all     = X_all.drop(columns=TE_COLS)
Xtest_raw = Xtest_raw.drop(columns=TE_COLS)
for col in TE_COLS:
    for sfx in te_suffixes:
        X_all[f'TE_{sfx}_{col}']     = oof_te[f'TE_{sfx}_{col}']
        Xtest_raw[f'TE_{sfx}_{col}'] = test_te[f'TE_{sfx}_{col}']
print(f'TE done in {time.time()-t0:.0f}s. Feature count: {X_all.shape[1]}')

# LightGBM — 3 seeds
LGB_SEEDS = [42, 123, 456]
lgb_params = {
    'objective':'binary','metric':'auc','n_estimators':2000,'learning_rate':0.05,
    'num_leaves':127,'subsample':0.8,'colsample_bytree':0.6,'min_child_samples':20,
    'reg_alpha':0.1,'reg_lambda':1.0,'n_jobs':-1,'verbose':-1
}

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof_lgb  = np.zeros(len(X_all),     dtype=np.float64)
pred_lgb = np.zeros(len(Xtest_raw), dtype=np.float64)

for seed in LGB_SEEDS:
    print(f'\n=== LightGBM seed={seed} ===')
    oof_s = np.zeros(len(X_all),     dtype=np.float64)
    tp_s  = np.zeros(len(Xtest_raw), dtype=np.float64)
    for fold, (tri, vali) in enumerate(skf.split(X_all, y_all), 1):
        fold_start = time.time()
        Xtr = X_all.iloc[tri].copy(); Xval = X_all.iloc[vali].copy()
        ytr = y_all.iloc[tri].copy(); yval = y_all.iloc[vali].copy()
        Xte = Xtest_raw.copy()
        for c in CATS:
            if c in Xtr.columns: Xtr[c], Xval[c], Xte[c] = factorize3(Xtr[c], Xval[c], Xte[c])
        for c in Xtr.select_dtypes('object').columns:
            Xtr[c], Xval[c], Xte[c] = factorize3(Xtr[c], Xval[c], Xte[c])
        model = lgb.LGBMClassifier(**{**lgb_params, 'random_state': seed})
        model.fit(Xtr, ytr, eval_set=[(Xval, yval)],
                  callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(False)])
        oof_s[vali] = model.predict_proba(Xval)[:, 1]
        tp_s += model.predict_proba(Xte)[:, 1] / N_SPLITS
        print(f'  Fold {fold} AUC: {roc_auc_score(yval, oof_s[vali]):.5f}  '
              f'iter={model.best_iteration_}  time={time.time()-fold_start:.0f}s')
    print(f'  Seed {seed} OOF: {roc_auc_score(y_all, oof_s):.5f}')
    oof_lgb  += oof_s / len(LGB_SEEDS)
    pred_lgb += tp_s  / len(LGB_SEEDS)

print(f'\nLightGBM OOF AUC: {roc_auc_score(y_all, oof_lgb):.5f}')
np.save('oof_lgb.npy', oof_lgb); np.save('pred_lgb.npy', pred_lgb)

In [ ]:
# XGBoost — 3 seeds
XGB_SEEDS = [42, 123, 456]
xgb_params = {
    'objective': 'binary:logistic', 'eval_metric': 'auc',
    'n_estimators': 5000, 'learning_rate': 0.0063,
    'max_depth': 5, 'colsample_bytree': 0.32,
    'min_child_weight': 6, 'reg_alpha': 3.5, 'reg_lambda': 1.3,
    'gamma': 0.79, 'subsample': 0.81,
    'tree_method': 'hist', 'n_jobs': -1, 'verbosity': 0,
    'early_stopping_rounds': 500,
}

oof_xgb  = np.zeros(len(X_all),     dtype=np.float64)
pred_xgb = np.zeros(len(Xtest_raw), dtype=np.float64)

for seed in XGB_SEEDS:
    print(f'\n=== XGBoost seed={seed} ===')
    oof_s = np.zeros(len(X_all),     dtype=np.float64)
    tp_s  = np.zeros(len(Xtest_raw), dtype=np.float64)
    for fold, (tri, vali) in enumerate(skf.split(X_all, y_all), 1):
        fold_start = time.time()
        Xtr = X_all.iloc[tri].copy(); Xval = X_all.iloc[vali].copy()
        ytr = y_all.iloc[tri].copy(); yval = y_all.iloc[vali].copy()
        Xte = Xtest_raw.copy()
        for c in CATS:
            if c in Xtr.columns: Xtr[c], Xval[c], Xte[c] = factorize3(Xtr[c], Xval[c], Xte[c])
        for c in Xtr.select_dtypes('object').columns:
            Xtr[c], Xval[c], Xte[c] = factorize3(Xtr[c], Xval[c], Xte[c])
        model_xgb = xgb.XGBClassifier(**{**xgb_params, 'random_state': seed})
        model_xgb.fit(Xtr, ytr, eval_set=[(Xval, yval)], verbose=False)
        oof_s[vali] = model_xgb.predict_proba(Xval)[:, 1]
        tp_s += model_xgb.predict_proba(Xte)[:, 1] / N_SPLITS
        print(f'  Fold {fold} AUC: {roc_auc_score(yval, oof_s[vali]):.5f}  '
              f'iter={model_xgb.best_iteration}  time={time.time()-fold_start:.0f}s')
    print(f'  Seed {seed} OOF: {roc_auc_score(y_all, oof_s):.5f}')
    oof_xgb  += oof_s / len(XGB_SEEDS)
    pred_xgb += tp_s  / len(XGB_SEEDS)

print(f'\nXGBoost OOF AUC: {roc_auc_score(y_all, oof_xgb):.5f}')
np.save('oof_xgb.npy', oof_xgb); np.save('pred_xgb.npy', pred_xgb)

In [ ]:
from catboost import CatBoostClassifier

# CatBoost uses raw categoricals natively — no TE needed
# Include BASE + ORIG + ARITH + SERVICE + FREQ + DIGIT + ROUND (exclude TE_COLS strings)
CAT_FEATURES = [c for c in FEATURES if c not in TE_COLS]
cat_cols_cb  = [c for c in CATS if c in CAT_FEATURES]
cat_idx_cb   = [CAT_FEATURES.index(c) for c in cat_cols_cb]

cb_params = dict(
    iterations=2000, learning_rate=0.05, depth=6,
    l2_leaf_reg=3, eval_metric='Logloss', od_type='Iter', od_wait=100,
    verbose=False, task_type='GPU', devices='0',
    auto_class_weights='Balanced',
)

# Single seed only — CatBoost on GPU is slow per fold, 1 seed keeps runtime manageable
oof_cat  = np.zeros(len(X_all),     dtype=np.float64)
pred_cat = np.zeros(len(Xtest_raw), dtype=np.float64)
cv_start = time.time()

for fold, (tri, vali) in enumerate(skf.split(X_all, y_all), 1):
    fold_start = time.time()
    Xtr  = X_all[CAT_FEATURES].iloc[tri].copy()
    Xval = X_all[CAT_FEATURES].iloc[vali].copy()
    ytr  = y_all.iloc[tri].copy(); yval = y_all.iloc[vali].copy()
    Xte  = Xtest_raw[CAT_FEATURES].copy()
    model_cb = CatBoostClassifier(**{**cb_params, 'random_seed': SEED})
    model_cb.fit(Xtr, ytr, cat_features=cat_idx_cb,
                 eval_set=(Xval, yval), use_best_model=True)
    oof_cat[vali] = model_cb.predict_proba(Xval)[:, 1]
    pred_cat += model_cb.predict_proba(Xte)[:, 1] / N_SPLITS
    print(f'CatBoost Fold {fold} AUC: {roc_auc_score(yval, oof_cat[vali]):.5f}  '
          f'best_iter={model_cb.best_iteration_}  time={time.time()-fold_start:.0f}s')

print(f'\nCatBoost OOF AUC: {roc_auc_score(y_all, oof_cat):.5f}  total={time.time()-cv_start:.0f}s')
np.save('oof_cat.npy', oof_cat); np.save('pred_cat.npy', pred_cat)

In [ ]:
NUM_COLS_BASE = ['tenure', 'MonthlyCharges', 'TotalCharges']

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# NN uses 16 raw categoricals (learned embeddings) + 12 scaled numerics
nn_cat_cols = CATS
nn_num_cols = NUM_COLS_BASE + ARITH + SERVICE + FREQ  # 3 + 3 + 3 + 3 = 12
print(f'NN numeric features: {len(nn_num_cols)}')

cat_vocabs = {}
for c in nn_cat_cols:
    all_vals = pd.concat([X_all[c], Xtest_raw[c]], ignore_index=True).astype(str)
    cat_vocabs[c] = {v: i+1 for i, v in enumerate(all_vals.unique())}

def encode_cats(df, vocabs):
    return {c: df[c].astype(str).map(vocab).fillna(0).astype('int64')
            for c, vocab in vocabs.items()}

class ChurnMLP(nn.Module):
    def __init__(self, cat_vocabs, num_dim, hidden=[256, 128, 64], dropout=0.4):
        super().__init__()
        self.cat_names = list(cat_vocabs.keys())
        self.embeddings = nn.ModuleDict({
            c: nn.Embedding(len(v)+1, min(50, (len(v)+1)//2))
            for c, v in cat_vocabs.items()
        })
        emb_total = sum(min(50, (len(v)+1)//2) for v in cat_vocabs.values())
        in_dim = emb_total + num_dim
        layers = [nn.BatchNorm1d(in_dim), nn.Dropout(0.2)]
        for h in hidden:
            layers += [nn.Linear(in_dim, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        layers += [nn.Linear(in_dim, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x_cat, x_num):
        embs = [self.embeddings[c](x_cat[:, i]) for i, c in enumerate(self.cat_names)]
        return self.net(torch.cat(embs + [x_num], dim=1)).squeeze(1)

def make_tensors(Xdf, yvals=None):
    cat_enc = encode_cats(Xdf, cat_vocabs)
    x_cat = torch.tensor(
        np.stack([cat_enc[c].values for c in nn_cat_cols], axis=1), dtype=torch.long
    ).to(device)
    x_num = torch.tensor(Xdf[nn_num_cols].values.astype('float32')).to(device)
    if yvals is not None:
        return x_cat, x_num, torch.tensor(yvals.values.astype('float32')).to(device)
    return x_cat, x_num

oof_nn  = np.zeros(len(X_all),     dtype=np.float32)
pred_nn = np.zeros(len(Xtest_raw), dtype=np.float32)
cv_start = time.time()

for fold, (tri, vali) in enumerate(skf.split(X_all, y_all), 1):
    fold_start = time.time()
    Xtr  = X_all.iloc[tri].copy();  Xval = X_all.iloc[vali].copy()
    ytr  = y_all.iloc[tri].copy();  yval = y_all.iloc[vali].copy()
    Xte  = Xtest_raw.copy()

    scaler = StandardScaler()
    Xtr[nn_num_cols]  = scaler.fit_transform(Xtr[nn_num_cols].values.astype('float32'))
    Xval[nn_num_cols] = scaler.transform(Xval[nn_num_cols].values.astype('float32'))
    Xte[nn_num_cols]  = scaler.transform(Xte[nn_num_cols].values.astype('float32'))

    xc_tr, xn_tr, y_tr = make_tensors(Xtr, ytr)
    xc_val, xn_val, _  = make_tensors(Xval, yval)
    xc_te, xn_te       = make_tensors(Xte)

    loader = DataLoader(TensorDataset(xc_tr, xn_tr, y_tr), batch_size=2048, shuffle=True)
    model_nn = ChurnMLP(cat_vocabs, len(nn_num_cols)).to(device)
    optimizer = torch.optim.Adam(model_nn.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=150)
    criterion = nn.BCEWithLogitsLoss()

    best_auc_nn = 0; patience_ct = 0; best_state = None
    for epoch in range(150):
        model_nn.train()
        for xc_b, xn_b, y_b in loader:
            optimizer.zero_grad()
            criterion(model_nn(xc_b, xn_b), y_b).backward()
            optimizer.step()
        scheduler.step()
        model_nn.eval()
        with torch.no_grad():
            val_auc = roc_auc_score(yval, torch.sigmoid(model_nn(xc_val, xn_val)).cpu().numpy())
        if val_auc > best_auc_nn:
            best_auc_nn = val_auc; patience_ct = 0
            best_state = {k: v.clone() for k, v in model_nn.state_dict().items()}
        else:
            patience_ct += 1
            if patience_ct >= 15: break

    model_nn.load_state_dict(best_state)
    model_nn.eval()
    with torch.no_grad():
        oof_nn[vali] = torch.sigmoid(model_nn(xc_val, xn_val)).cpu().numpy()
        pred_nn += torch.sigmoid(model_nn(xc_te, xn_te)).cpu().numpy() / N_SPLITS

    print(f'NN Fold {fold} AUC: {roc_auc_score(yval, oof_nn[vali]):.5f}  '
          f'epochs={epoch-patience_ct+1}  time={time.time()-fold_start:.0f}s')

print(f'\nNN OOF AUC: {roc_auc_score(y_all, oof_nn):.5f}  total={time.time()-cv_start:.0f}s')
np.save('oof_nn.npy', oof_nn); np.save('pred_nn.npy', pred_nn)

In [ ]:
# ── Ensemble: COBYLA optimizer + Hill Climbing fallback ───────
models = {
    'lgb': (oof_lgb,  pred_lgb),
    'xgb': (oof_xgb,  pred_xgb),
    'cat': (oof_cat,  pred_cat),
    'nn':  (oof_nn,   pred_nn),
}

print('Single model OOF AUCs:')
for name, (o, _) in models.items():
    print(f'  {name}: {roc_auc_score(y_all, o):.5f}')

# ── COBYLA optimizer (primary) ────────────────────────────────
n = len(models)
model_names = list(models.keys())
oof_stack  = np.column_stack([models[k][0] for k in model_names])
pred_stack = np.column_stack([models[k][1] for k in model_names])

def neg_auc(w):
    return -roc_auc_score(y_all, oof_stack @ w)

constraints = [{'type': 'eq', 'fun': lambda w: w.sum() - 1}]
bounds_list = [(0, 1)] * n
result = minimize(neg_auc, x0=np.ones(n)/n, method='COBYLA',
                  constraints=constraints,
                  options={'maxiter': 10000, 'rhobeg': 0.1})
cobyla_w = np.clip(result.x, 0, 1)
cobyla_w /= cobyla_w.sum()
cobyla_auc = roc_auc_score(y_all, oof_stack @ cobyla_w)
print(f'\nCOBYLA weights: {dict(zip(model_names, cobyla_w.round(4)))}')
print(f'COBYLA OOF AUC: {cobyla_auc:.5f}')

# ── Hill climbing (fallback) ─────────────────────────────────
best_name = max(models, key=lambda k: roc_auc_score(y_all, models[k][0]))
best_auc  = roc_auc_score(y_all, models[best_name][0])
weights_hc = {best_name: 1.0}
blend_oof_hc  = models[best_name][0].copy().astype(float)
blend_pred_hc = models[best_name][1].copy().astype(float)

TOL = 1e-5
for step in range(30):
    best_step_auc = best_auc
    best_w = None; best_add = None
    for name, (o, _) in models.items():
        for w in np.arange(0.05, 1.0, 0.05):
            trial = (blend_oof_hc + w * o) / (1 + w)
            auc = roc_auc_score(y_all, trial)
            if auc > best_step_auc:
                best_step_auc = auc; best_w = w; best_add = name
        for w in np.arange(0.05, 0.5, 0.05):
            trial = np.clip((blend_oof_hc - w * o) / (1 - w), 0, 1)
            auc = roc_auc_score(y_all, trial)
            if auc > best_step_auc:
                best_step_auc = auc; best_w = -w; best_add = name
    if best_add is None or (best_step_auc - best_auc) < TOL:
        break
    w = best_w
    blend_oof_hc  = np.clip((blend_oof_hc  + w * models[best_add][0]) / (1 + w), 0, 1)
    blend_pred_hc = np.clip((blend_pred_hc + w * models[best_add][1]) / (1 + w), 0, 1)
    weights_hc[best_add] = weights_hc.get(best_add, 0) + w
    best_auc = best_step_auc

hc_auc = roc_auc_score(y_all, blend_oof_hc)
print(f'\nHill-climb weights: {dict((k, round(v, 3)) for k, v in weights_hc.items())}')
print(f'Hill-climb OOF AUC: {hc_auc:.5f}')

# ── Pick whichever is better ─────────────────────────────────
if cobyla_auc >= hc_auc:
    blend_oof  = oof_stack @ cobyla_w
    blend_pred = pred_stack @ cobyla_w
    print(f'\n>>> Using COBYLA (AUC={cobyla_auc:.5f})')
else:
    blend_oof  = blend_oof_hc
    blend_pred = blend_pred_hc
    print(f'\n>>> Using Hill-climb (AUC={hc_auc:.5f})')

print(f'Final ensemble OOF AUC: {roc_auc_score(y_all, blend_oof):.5f}')
sub['Churn'] = blend_pred
sub.to_csv('submission_ensemble.csv', index=False)
print('Saved submission_ensemble.csv')

In [ ]:
# ── Pseudo-labeling ──────────────────────────────────────────
PSEUDO_THRESH = 0.15
mask = (blend_pred < PSEUDO_THRESH) | (blend_pred > (1 - PSEUDO_THRESH))
print(f'High-confidence test samples: {mask.sum():,} / {len(blend_pred):,} ({mask.mean()*100:.1f}%)')

X_pseudo = pd.concat([X_all, Xtest_raw[mask].reset_index(drop=True)], ignore_index=True)
y_pseudo  = np.concatenate([y_all.values.astype('float32'), blend_pred[mask].astype('float32')])
print(f'Expanded training set: {len(X_pseudo):,} rows')

n_est = 500
pseudo_params = {
    'objective': 'binary', 'n_estimators': n_est,
    'learning_rate': 0.05, 'num_leaves': 127,
    'subsample': 0.8, 'colsample_bytree': 0.6,
    'min_child_samples': 20, 'reg_alpha': 0.1, 'reg_lambda': 1.0,
    'n_jobs': -1, 'verbose': -1,
}

pred_pseudo = np.zeros(len(Xtest_raw), dtype=np.float64)
for seed in LGB_SEEDS:
    Xp = X_pseudo.copy(); Xte = Xtest_raw.copy()
    for c in CATS:
        if c in Xp.columns:
            codes, _ = pd.factorize(pd.concat([Xp[c], Xte[c]], ignore_index=True).astype(str))
            Xp[c] = codes[:len(Xp)].astype('int32'); Xte[c] = codes[len(Xp):].astype('int32')
    for c in Xp.select_dtypes('object').columns:
        codes, _ = pd.factorize(pd.concat([Xp[c], Xte[c]], ignore_index=True).astype(str))
        Xp[c] = codes[:len(Xp)].astype('int32'); Xte[c] = codes[len(Xp):].astype('int32')
    m = lgb.LGBMRegressor(**{**pseudo_params, 'random_state': seed})
    m.fit(Xp, y_pseudo)
    pred_pseudo += np.clip(m.predict(Xte), 0, 1) / len(LGB_SEEDS)
    print(f'  Seed {seed} done')

PSEUDO_WEIGHT = 0.3
final_pred = (1 - PSEUDO_WEIGHT) * blend_pred + PSEUDO_WEIGHT * pred_pseudo
sub['Churn'] = final_pred
sub.to_csv('submission_final.csv', index=False)
print(f'Saved submission_final.csv')